In [0]:
#customers_df = spark.read.table("customer")
customers_df = spark.table("customer")
display(customers_df.select("customer_id","email"))

In [0]:
from pyspark.sql.functions import col
display(customers_df.select(col("customer_id"),col("email").alias("email_address"), col("updated").cast("timestamp"))) 

In [0]:
display(customers_df.drop("profile", "updated"))

In [0]:
from pyspark.sql.functions import col, split

new_customers_df = customers_df.withColumn("customer_domain", split(col("email"), "@").getItem(1)) \
    .withColumnRenamed("email", "customer_email") \
    .drop("profile", "updated")
 
display(new_customers_df)

# new_customers_df = (
#                     customers_df.withColumn("customer_domain", F.split(F.col("email"), "@").getItem(1))
#                                 .withColumnRenamed("email", "customer_email")
#                                 .drop("profile", "updated")
#                     )

In [0]:
orders_df = spark.table("orders")

display(orders_df.describe())
#display(orders_df.summary())

## Data Deduplication

In [0]:
orders_df = spark.table("orders")
display(orders_df)

In [0]:
%sql
INSERT INTO orders
VALUES ('C00025', NULL, 'O00001', 7, NULL, 100.0, null)

In [0]:
#deduped_df = orders_df.distinct()
#deduped_df = orders_df.dropDuplicates()

deduped_df = orders_df.dropDuplicates(["order_id","customer_id"])
display(deduped_df)

## Null Handling

In [0]:
%sql
INSERT INTO customer(customer_id)
VALUES ("C00031");

INSERT INTO customer(customer_id)
VALUES ("C00032");

In [0]:
customers_df = spark.table("customer")
display(customers_df)

In [0]:
#clean_df = customers_df.fillna({"email": "unknown@example.com"})
clean_df = customers_df.na.fill({"email": "unknown@example.com"})
display(clean_df)

In [0]:
#clean_df = customers_df.dropna()
clean_df = customers_df.na.drop(subset=["first_name"])
display(clean_df)

## Join

In [0]:
joined_df = customers_df.join(orders_df, "customer_id", "left")
display(joined_df)

In [0]:
anti_joined_df = customers_df.join(orders_df, "customer_id", "left_anti")
display(anti_joined_df)

In [0]:
display(orders_df.where("customer_id = 'C00005'"))

In [0]:
motocycles_df = spark.table("motocycles")
orders_df = spark.table("orders")

In [0]:
from pyspark.sql.functions import explode, broadcast

exploded_orders_df = (orders_df.withColumn("motocycle", explode("motocycles"))
                                .select("*", "motocycle.motocycle_id", "motocycle.subtotal")
                                .drop("motocycles", "motocycle")
                     )

orders_details_df = exploded_orders_df.join(broadcast(motocycles_df), "motocycle_id", "inner")
display(orders_details_df)

## 
Querying Data

In [0]:
customers_details_df = spark.sql("""SELECT customer_id, email, updated,
                                        profile.first_name,
                                        profile.last_name,
                                        profile.gender,
                                        profile.adresse.street,
                                        profile.adresse.city,
                                        profile.adresse.country
                                    FROM customer
                                """)

display(customers_details_df)

In [0]:
customers_details_df.write.mode("overwrite").saveAsTable("customers_details")

In [0]:
%sql
SELECT * FROM customers_details